# Week 13: Mini Project v2.1 — Improve & Extend — PHASE 7: Proving Mastery

*Core Mastery: "I can parameterize, extend, and thoroughly test a working pipeline"*

*Computer Programming II | 5 Hours | Dr. Arif Solmaz*

## 🎯 Learning Objectives

1. Centralize pipeline parameters in a CONFIG cell
2. Implement a moving average filter with configurable window size
3. Implement a median filter and compare with moving average
4. Create side-by-side comparison plots of different filters
5. Write 15+ assertion-based tests across all pipeline stages
6. Extend the pipeline to handle N sensor IDs generically
7. Generate a formatted text summary report
8. Integrate all improvements into a CONFIG-driven `main()` function
9. Understand trade-offs between smoothing filters
10. Practice systematic testing methodology

## 🎯 Core Mastery Connection

A working pipeline is only the first step. Real engineering means making the code configurable, robust, and well-tested. This week you will refactor the Week 12 pipeline by extracting hard-coded values into a CONFIG cell, adding two smoothing filters (moving average and median), writing comprehensive tests, and producing a summary report. By the end you will have a production-quality pipeline.

---
## 🧭 Five-Hour Class Roadmap

This notebook is designed for one five-hour class with four short breaks.

| Target | Activity |
|---|---|
| 00:00–00:55 | Concepts and examples → Checkpoint 1 |
| 00:55–01:05 | Break |
| 01:05–01:55 | Concepts and examples → Checkpoint 2 |
| 01:55–02:05 | Break |
| 02:05–02:55 | Concepts and examples → Checkpoint 3 |
| 02:55–03:05 | Break |
| 03:05–03:55 | Concepts and examples → Checkpoint 4 |
| 03:55–04:05 | Break |
| 04:05–04:45 | Core Practice (Exercises 1–8) → Checkpoint 5 |
| 04:45–05:00 | Review and retry failed checks |

Checkpoints provide immediate feedback only inside your Colab runtime. Nothing
is transmitted, saved for grading, or reviewed by the instructor. Exercises 9
and above are optional extensions—not homework.


In [ ]:
# Run this setup cell once at the start of class.
_checkpoint_results = {}

def check_answer(number, answer, expected, explanation):
    actual = str(answer).strip().lower().replace(" ", "")
    target = str(expected).strip().lower().replace(" ", "")
    correct = actual == target
    _checkpoint_results[int(number)] = (int(correct), 1)
    if correct:
        print(f"✅ Checkpoint {number}: correct")
        print("Why:", explanation)
    elif not str(answer).strip():
        print(f"🟡 Checkpoint {number}: enter an answer, then run this cell again.")
    else:
        print(f"🔴 Checkpoint {number}: not yet. Review the preceding examples and retry.")
    return correct

def exercise_checkpoint(number, expected=8):
    """Count core exercise cells that contain work and were run in this runtime."""
    import re
    completed = set()
    for source in globals().get("In", []):
        match = re.search(r"#\s*✏️\s*\[EX(\d+)\]", str(source), flags=re.I)
        if not match:
            continue
        answer = re.sub(r"^.*?#\s*✏️\s*\[EX\d+\]", "", str(source), count=1, flags=re.I | re.S).strip()
        if answer and answer != "pass" and "your code here" not in answer.lower():
            completed.add(int(match.group(1)))
    checks = [(index, index in completed) for index in range(1, expected + 1)]
    passed = sum(done for _, done in checks)
    _checkpoint_results[int(number)] = (passed, expected)
    print(f"Core Practice: {passed}/{expected} exercise cells edited and run")
    missing = [str(index) for index, done in checks if not done]
    if not missing:
        print("✅ Core Practice complete.")
    else:
        print("🟡 Still to complete/run:", ", ".join(missing))
    return passed, expected

def show_progress_summary():
    print("\n=== My local progress ===")
    for number in range(1, 6):
        if number in _checkpoint_results:
            passed, total = _checkpoint_results[number]
            print(f"Checkpoint {number}: {passed}/{total}")
        else:
            print(f"Checkpoint {number}: not run")
    print("Results exist only in this temporary runtime.")

print("✅ Local self-check tools ready")


---
## Part 1: CONFIG Cell

All pipeline parameters live in one place. This makes it easy to change behavior without editing function code.

| Parameter | Type | Description |
|-----------|------|-------------|
| `DATA_CSV` | str | Raw CSV data (inline or file path) |
| `THRESHOLD` | float | Outlier threshold multiplier |
| `WINDOW_SIZE` | int | Smoothing window size |
| `OUTPUT_DIR` | str | Directory for output files |

**Figure 1.1** — CONFIG cell definition

In [ ]:
# ━━━━━━━━━━ CONFIG ━━━━━━━━━━
CONFIG = {
    "DATA_CSV": """timestamp,sensor_id,value,status
2025-06-01 08:00,S01,23.5,OK
2025-06-01 08:00,S02,45.1,OK
2025-06-01 08:00,S03,12.8,OK
2025-06-01 09:00,S01,24.1,OK
2025-06-01 09:00,S02,,WARN
2025-06-01 09:00,S03,13.2,OK
2025-06-01 10:00,S01,9999,ERROR
2025-06-01 10:00,S02,47.3,OK
2025-06-01 10:00,S03,11.9,OK
2025-06-01 11:00,S01,25.0,OK
2025-06-01 11:00,S02,44.8,OK
2025-06-01 11:00,S03,,
2025-06-01 12:00,S01,23.8,OK
2025-06-01 12:00,S02,46.5,WARN
2025-06-01 12:00,S03,14.1,OK
2025-06-01 13:00,S01,24.9,OK
2025-06-01 13:00,S02,9999,OK
2025-06-01 13:00,S03,13.7,OK
2025-06-01 14:00,S01,22.1,INVALID
2025-06-01 14:00,S02,43.2,OK
2025-06-01 14:00,S03,15.0,OK
2025-06-01 15:00,S01,26.3,OK
2025-06-01 15:00,S02,48.0,OK
2025-06-01 15:00,S03,12.5,OK""",
    "THRESHOLD": 1.5,
    "WINDOW_SIZE": 3,
    "OUTPUT_DIR": "/tmp/sensor_output_v2"
}
print("CONFIG loaded:")
for k, v in CONFIG.items():
    if k != "DATA_CSV":
        print(f"  {k} = {v}")
    else:
        print(f"  DATA_CSV = <{len(v)} chars>")

**Figure 1.2** — Using CONFIG in pipeline functions

In [ ]:
import io, csv, math, os

def read_sensor_data(csv_string):
    reader = csv.DictReader(io.StringIO(csv_string.strip()))
    records = []
    for row in reader:
        try:
            row["value"] = float(row["value"]) if row["value"].strip() else None
        except ValueError:
            row["value"] = None
        records.append(row)
    return records

def validate_schema(records):
    required = {"timestamp", "sensor_id", "value", "status"}
    if not records:
        raise ValueError("No records")
    missing = required - set(records[0].keys())
    if missing:
        raise ValueError(f"Missing columns: {missing}")
    for r in records:
        if not r.get("status") or r["status"].strip() == "":
            r["status"] = "UNKNOWN"
    return records

def clean_missing(records):
    return [r for r in records if r["value"] is not None]

def remove_invalid(records):
    return [r for r in records if r["status"] in {"OK", "WARN", "ERROR"}]

def filter_outliers(records, threshold=1.5):
    values = sorted(r["value"] for r in records)
    n = len(values)
    q1, q3 = values[n // 4], values[3 * n // 4]
    iqr = q3 - q1
    lower, upper = q1 - threshold * iqr, q3 + threshold * iqr
    return [r for r in records if lower <= r["value"] <= upper]

# Quick test with CONFIG
data = read_sensor_data(CONFIG["DATA_CSV"])
data = validate_schema(data)
data = clean_missing(data)
data = remove_invalid(data)
data = filter_outliers(data, CONFIG["THRESHOLD"])
print(f"After cleaning with threshold={CONFIG['THRESHOLD']}: {len(data)} records")

---
## Part 2: Moving Average Filter

A moving average smooths noisy data by averaging each point with its neighbors.

**Formula:** For window size $w$ and data point $x_i$:
$$MA_i = \\frac{1}{w} \\sum_{j=i-(w-1)/2}^{i+(w-1)/2} x_j$$

Edge values use smaller windows (as many neighbors as available).

**Figure 2.1** — `moving_average()` function

In [ ]:
def moving_average(data, window):
    """Apply moving average filter to a list of numbers."""
    result = []
    half = window // 2
    for i in range(len(data)):
        start = max(0, i - half)
        end = min(len(data), i + half + 1)
        avg = sum(data[start:end]) / (end - start)
        result.append(round(avg, 4))
    return result

# Test
test_values = [10, 12, 11, 15, 14, 13, 16, 12, 11, 10]
smoothed = moving_average(test_values, 3)
print("Original:", test_values)
print("MA(3):   ", smoothed)

**Figure 2.2** — Moving average on sensor data

In [ ]:
def apply_moving_average(records, window):
    """Apply moving average per sensor."""
    groups = {}
    for r in records:
        sid = r["sensor_id"]
        if sid not in groups:
            groups[sid] = []
        groups[sid].append(r)

    result = []
    for sid in sorted(groups.keys()):
        group = groups[sid]
        values = [r["value"] for r in group]
        smoothed = moving_average(values, window)
        for r, sv in zip(group, smoothed):
            r_copy = dict(r)
            r_copy["value_ma"] = sv
            result.append(r_copy)
        print(f"  Sensor {sid}: applied MA({window}) to {len(group)} values")
    return result

ma_data = apply_moving_average(data, CONFIG["WINDOW_SIZE"])
print(f"\nTotal records with MA: {len(ma_data)}")

**Figure 2.3** — Visualizing moving average effect

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# Plot for first sensor
s01 = [r for r in ma_data if r["sensor_id"] == "S01"]
raw_vals = [r["value"] for r in s01]
ma_vals = [r["value_ma"] for r in s01]

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(raw_vals, "o-", label="Raw", alpha=0.6)
ax.plot(ma_vals, "s-", label=f"MA({CONFIG['WINDOW_SIZE']})", linewidth=2)
ax.set_title("Sensor S01: Raw vs Moving Average")
ax.set_xlabel("Reading Index")
ax.set_ylabel("Value")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
print("Moving average plot created for S01")

---
### ⏱️ Checkpoint 1 of 5 — Configuration (target 00:55)

Should thresholds be centralized or scattered?

Enter a short answer in the next cell and run it. Retry after reviewing the
preceding examples if needed.


In [ ]:
checkpoint_1_answer = ""  # enter your answer
check_answer(
    1, checkpoint_1_answer, 'centralized',
    'A CONFIG section makes changes controlled and visible.',
)


---
## Part 3: Median Filter

A median filter replaces each point with the median of its window. It is better than moving average at preserving edges and removing spike outliers.

| Filter | Strengths | Weaknesses |
|--------|-----------|------------|
| Moving Average | Smooth output | Blurs edges, affected by outliers |
| Median Filter | Preserves edges, removes spikes | Slower computation |

**Figure 3.1** — `median_filter()` function

In [ ]:
def median_filter(data, window):
    """Apply median filter to a list of numbers."""
    result = []
    half = window // 2
    for i in range(len(data)):
        start = max(0, i - half)
        end = min(len(data), i + half + 1)
        chunk = sorted(data[start:end])
        mid = len(chunk) // 2
        if len(chunk) % 2 == 0:
            median = (chunk[mid - 1] + chunk[mid]) / 2
        else:
            median = chunk[mid]
        result.append(round(median, 4))
    return result

# Test with spike
spike_data = [10, 11, 100, 12, 11, 10, 13, 11, 200, 12]
print("Original:      ", spike_data)
print("MA(3):         ", moving_average(spike_data, 3))
print("Median(3):     ", median_filter(spike_data, 3))

**Figure 3.2** — Applying median filter to sensor data

In [ ]:
def apply_median_filter(records, window):
    """Apply median filter per sensor."""
    groups = {}
    for r in records:
        sid = r["sensor_id"]
        if sid not in groups:
            groups[sid] = []
        groups[sid].append(r)

    result = []
    for sid in sorted(groups.keys()):
        group = groups[sid]
        values = [r["value"] for r in group]
        filtered = median_filter(values, window)
        for r, fv in zip(group, filtered):
            r_copy = dict(r)
            r_copy["value_median"] = fv
            result.append(r_copy)
        print(f"  Sensor {sid}: applied Median({window}) to {len(group)} values")
    return result

med_data = apply_median_filter(data, CONFIG["WINDOW_SIZE"])
print(f"Total records with median filter: {len(med_data)}")

**Figure 3.3** — Comparison table: raw vs MA vs median

In [ ]:
# Merge MA and Median for S01
s01_raw = [r["value"] for r in data if r["sensor_id"] == "S01"]
s01_ma = moving_average(s01_raw, CONFIG["WINDOW_SIZE"])
s01_med = median_filter(s01_raw, CONFIG["WINDOW_SIZE"])

print(f"{'Index':>6} {'Raw':>8} {'MA':>8} {'Median':>8}")
print("-" * 34)
for i in range(len(s01_raw)):
    print(f"{i:>6} {s01_raw[i]:8.2f} {s01_ma[i]:8.4f} {s01_med[i]:8.4f}")

---
## Part 4: Comparing Filters

Side-by-side visualization is the best way to understand filter behavior.

**Figure 4.1** — Three-line comparison plot

In [ ]:
def plot_filter_comparison(raw, ma, med, title="Filter Comparison"):
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(raw, "o-", label="Raw", alpha=0.5)
    ax.plot(ma, "s-", label="Moving Avg", linewidth=2)
    ax.plot(med, "^-", label="Median", linewidth=2)
    ax.set_title(title)
    ax.set_xlabel("Reading Index")
    ax.set_ylabel("Value")
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    return fig

fig = plot_filter_comparison(s01_raw, s01_ma, s01_med, "S01: Raw vs MA vs Median")
print("Filter comparison plot created")

**Figure 4.2** — Multi-sensor comparison subplots

In [ ]:
def plot_all_sensors_comparison(records, window):
    groups = {}
    for r in records:
        sid = r["sensor_id"]
        if sid not in groups:
            groups[sid] = []
        groups[sid].append(r["value"])

    sids = sorted(groups.keys())
    fig, axes = plt.subplots(len(sids), 1, figsize=(10, 4 * len(sids)))
    if len(sids) == 1:
        axes = [axes]
    for ax, sid in zip(axes, sids):
        raw = groups[sid]
        ma = moving_average(raw, window)
        med = median_filter(raw, window)
        ax.plot(raw, "o-", label="Raw", alpha=0.5)
        ax.plot(ma, "s-", label="MA")
        ax.plot(med, "^-", label="Median")
        ax.set_title(f"Sensor {sid}")
        ax.legend()
        ax.grid(True, alpha=0.3)
    plt.tight_layout()
    return fig

fig_all = plot_all_sensors_comparison(data, CONFIG["WINDOW_SIZE"])
print("Multi-sensor comparison plot created")

**Figure 4.3** — Quantifying filter differences

In [ ]:
def filter_error(raw, filtered):
    """Compute mean absolute difference between raw and filtered."""
    return round(sum(abs(r - f) for r, f in zip(raw, filtered)) / len(raw), 4)

for sid_label, raw_vals in [("S01", s01_raw)]:
    ma_vals = moving_average(raw_vals, CONFIG["WINDOW_SIZE"])
    med_vals = median_filter(raw_vals, CONFIG["WINDOW_SIZE"])
    print(f"Sensor {sid_label}:")
    print(f"  MA  mean abs diff: {filter_error(raw_vals, ma_vals)}")
    print(f"  Med mean abs diff: {filter_error(raw_vals, med_vals)}")

---
### ⏱️ Checkpoint 2 of 5 — Moving average (target 01:55)

For valid mode, does a moving average shorten the sequence? Answer yes or no.

Enter a short answer in the next cell and run it. Retry after reviewing the
preceding examples if needed.


In [ ]:
checkpoint_2_answer = ""  # enter your answer
check_answer(
    2, checkpoint_2_answer, 'yes',
    'A complete window is required for each output.',
)


---
## Part 5: Expanding Tests

Systematic testing validates every pipeline stage. We use `assert` statements.

| Test Category | What to Test |
|---------------|-------------|
| `test_read()` | Correct row count, column names, type conversion |
| `test_clean()` | Missing removal, invalid removal, outlier removal |
| `test_normalize()` | Values in [0,1], per-sensor independence |
| `test_summary()` | Mean, min, max, count correctness |
| `test_filters()` | Window size effects, edge cases |

**Figure 5.1** — Read and validation tests

In [ ]:
def test_read():
    records = read_sensor_data(CONFIG["DATA_CSV"])
    assert len(records) == 24, f"Expected 24 rows, got {len(records)}"
    assert "timestamp" in records[0], "Missing timestamp column"
    assert "sensor_id" in records[0], "Missing sensor_id column"
    assert "value" in records[0], "Missing value column"
    assert "status" in records[0], "Missing status column"
    none_count = sum(1 for r in records if r["value"] is None)
    assert none_count == 2, f"Expected 2 None values, got {none_count}"
    print("test_read: 5 assertions passed")

def test_validate():
    records = read_sensor_data(CONFIG["DATA_CSV"])
    validated = validate_schema(records)
    assert len(validated) == 24
    unknown = [r for r in validated if r["status"] == "UNKNOWN"]
    assert len(unknown) >= 1, "Should have at least 1 UNKNOWN status"
    print("test_validate: 2 assertions passed")

test_read()
test_validate()

**Figure 5.2** — Cleaning and normalization tests

In [ ]:
def test_clean():
    records = read_sensor_data(CONFIG["DATA_CSV"])
    records = validate_schema(records)
    step1 = clean_missing(records)
    assert all(r["value"] is not None for r in step1), "Missing values remain"
    step2 = remove_invalid(step1)
    assert all(r["status"] in {"OK", "WARN", "ERROR"} for r in step2), "Invalid status remains"
    step3 = filter_outliers(step2, CONFIG["THRESHOLD"])
    assert len(step3) <= len(step2), "Outlier filter should not add records"
    assert len(step3) > 0, "Outlier filter removed all records"
    print("test_clean: 4 assertions passed")

def test_normalize():
    records = read_sensor_data(CONFIG["DATA_CSV"])
    records = validate_schema(records)
    records = clean_missing(records)
    records = remove_invalid(records)
    records = filter_outliers(records, CONFIG["THRESHOLD"])
    # Simple normalize
    groups = {}
    for r in records:
        sid = r["sensor_id"]
        if sid not in groups:
            groups[sid] = []
        groups[sid].append(r["value"])
    for sid, vals in groups.items():
        vmin, vmax = min(vals), max(vals)
        rng = vmax - vmin if vmax != vmin else 1
        normed = [(v - vmin) / rng for v in vals]
        assert all(0.0 <= v <= 1.0 for v in normed), f"Normalization out of bounds for {sid}"
    print("test_normalize: all sensor bounds verified")

test_clean()
test_normalize()

**Figure 5.3** — Filter and summary tests

In [ ]:
def test_filters():
    data = [10, 20, 15, 25, 12]
    ma = moving_average(data, 3)
    assert len(ma) == len(data), "MA length mismatch"
    assert ma[0] == round((10 + 20) / 2, 4), f"MA edge wrong: {ma[0]}"
    med = median_filter(data, 3)
    assert len(med) == len(data), "Median length mismatch"
    assert med[2] == 20, f"Median middle wrong: {med[2]}"
    # Window=1 should return original
    assert moving_average(data, 1) == data, "MA(1) should be identity"
    assert median_filter(data, 1) == data, "Median(1) should be identity"
    print("test_filters: 6 assertions passed")

def test_summary():
    records = [{"sensor_id": "T1", "value": 10}, {"sensor_id": "T1", "value": 20}, {"sensor_id": "T1", "value": 30}]
    groups = {"T1": [10, 20, 30]}
    mean = sum(groups["T1"]) / 3
    assert mean == 20.0, f"Mean wrong: {mean}"
    assert min(groups["T1"]) == 10
    assert max(groups["T1"]) == 30
    print("test_summary: 3 assertions passed")

test_filters()
test_summary()
print("\nAll 20 assertions passed!")

---
## Part 6: Multi-Sensor Support

The pipeline must work with any number of sensor IDs, not just S01-S03. We ensure all functions use generic grouping.

**Figure 6.1** — Generic grouping utility

In [ ]:
def group_by_sensor(records):
    """Group records by sensor_id — works for any number of sensors."""
    groups = {}
    for r in records:
        sid = r["sensor_id"]
        if sid not in groups:
            groups[sid] = []
        groups[sid].append(r)
    return groups

# Test with extra sensors
extra_csv = CONFIG["DATA_CSV"] + "\n2025-06-01 16:00,S04,55.5,OK\n2025-06-01 16:00,S05,60.2,OK"
extra = read_sensor_data(extra_csv)
extra = validate_schema(extra)
groups = group_by_sensor(extra)
print(f"Found {len(groups)} sensors: {sorted(groups.keys())}")
for sid, recs in sorted(groups.items()):
    print(f"  {sid}: {len(recs)} records")

**Figure 6.2** — Normalize and summarize for N sensors

In [ ]:
def normalize_per_sensor(records):
    groups = group_by_sensor(records)
    result = []
    for sid in sorted(groups.keys()):
        vals = [r["value"] for r in groups[sid] if r["value"] is not None]
        if not vals:
            continue
        vmin, vmax = min(vals), max(vals)
        rng = vmax - vmin if vmax != vmin else 1.0
        for r in groups[sid]:
            if r["value"] is not None:
                rc = dict(r)
                rc["value_raw"] = r["value"]
                rc["value"] = round((r["value"] - vmin) / rng, 6)
                result.append(rc)
    return result

def compute_summary(records, key="value_raw"):
    groups = group_by_sensor(records)
    summary = {}
    for sid in sorted(groups.keys()):
        vals = [r.get(key, r["value"]) for r in groups[sid]]
        n = len(vals)
        mean = sum(vals) / n
        var = sum((v - mean)**2 for v in vals) / n
        summary[sid] = {"mean": round(mean,2), "std": round(math.sqrt(var),2),
                        "min": round(min(vals),2), "max": round(max(vals),2), "count": n}
    return summary

clean = clean_missing(validate_schema(read_sensor_data(extra_csv)))
clean = remove_invalid(clean)
normed = normalize_per_sensor(clean)
stats = compute_summary(normed)
for sid, s in stats.items():
    print(f"{sid}: mean={s['mean']}, n={s['count']}")

**Figure 6.3** — Verifying N-sensor pipeline

In [ ]:
# Verify all sensors are represented
assert len(stats) == 5, f"Expected 5 sensors, got {len(stats)}"
for sid in ["S01","S02","S03","S04","S05"]:
    assert sid in stats, f"Missing sensor {sid}"
    assert stats[sid]["count"] > 0
print("Multi-sensor support verified for 5 sensors!")

---
### ⏱️ Checkpoint 3 of 5 — Robust filters (target 02:55)

Which resists spikes better: mean or median filter?

Enter a short answer in the next cell and run it. Retry after reviewing the
preceding examples if needed.


In [ ]:
checkpoint_3_answer = ""  # enter your answer
check_answer(
    3, checkpoint_3_answer, 'median filter',
    'A median is less affected by extreme values.',
)


---
## Part 7: Summary Report

A text report makes pipeline output human-readable.

**Figure 7.1** — `generate_report()` function

In [ ]:
def generate_report(summary, config):
    """Generate a formatted text report of pipeline results."""
    lines = []
    lines.append("=" * 50)
    lines.append("SENSOR LOG ANALYZER — SUMMARY REPORT")
    lines.append("=" * 50)
    lines.append(f"Window Size: {config['WINDOW_SIZE']}")
    lines.append(f"Outlier Threshold: {config['THRESHOLD']}")
    lines.append(f"Number of Sensors: {len(summary)}")
    lines.append("")
    lines.append(f"{'Sensor':>8} {'Mean':>8} {'Std':>8} {'Min':>8} {'Max':>8} {'N':>5}")
    lines.append("-" * 45)
    for sid in sorted(summary.keys()):
        s = summary[sid]
        lines.append(f"{sid:>8} {s['mean']:8.2f} {s['std']:8.2f} {s['min']:8.2f} {s['max']:8.2f} {s['count']:5d}")
    lines.append("")
    # Find highest and lowest mean
    best = max(summary, key=lambda s: summary[s]["mean"])
    worst = min(summary, key=lambda s: summary[s]["mean"])
    lines.append(f"Highest mean: {best} ({summary[best]['mean']})")
    lines.append(f"Lowest mean:  {worst} ({summary[worst]['mean']})")
    lines.append("=" * 50)
    return "\n".join(lines)

report = generate_report(stats, CONFIG)
print(report)

**Figure 7.2** — Saving report to file

In [ ]:
def save_report(report_text, filepath):
    """Save report text to file."""
    with open(filepath, "w") as f:
        f.write(report_text)
    print(f"Report saved to {filepath}")

os.makedirs(CONFIG["OUTPUT_DIR"], exist_ok=True)
save_report(report, os.path.join(CONFIG["OUTPUT_DIR"], "report.txt"))
print("Report generation complete")

---
## Part 8: Full Pipeline Integration

The complete v2.1 pipeline, driven by CONFIG.

**Figure 8.1** — Integrated `main()` function

In [ ]:
import json as json_mod

def main_v2(config):
    """Complete v2.1 pipeline driven by CONFIG."""
    os.makedirs(config["OUTPUT_DIR"], exist_ok=True)
    print("=" * 50)
    print("SENSOR LOG ANALYZER v2.1")
    print("=" * 50)

    # Read & Validate
    print("\n[1] Reading & validating...")
    records = read_sensor_data(config["DATA_CSV"])
    records = validate_schema(records)
    print(f"    {len(records)} records read")

    # Clean
    print("\n[2] Cleaning...")
    records = clean_missing(records)
    records = remove_invalid(records)
    records = filter_outliers(records, config["THRESHOLD"])
    print(f"    {len(records)} records after cleaning")

    # Apply filters
    print("\n[3] Applying filters...")
    groups = group_by_sensor(records)
    for sid in sorted(groups.keys()):
        vals = [r["value"] for r in groups[sid]]
        ma = moving_average(vals, config["WINDOW_SIZE"])
        med = median_filter(vals, config["WINDOW_SIZE"])
        for r, m, md_val in zip(groups[sid], ma, med):
            r["value_ma"] = m
            r["value_median"] = md_val
    all_records = []
    for sid in sorted(groups.keys()):
        all_records.extend(groups[sid])
    print(f"    Filters applied to {len(all_records)} records")

    # Normalize
    print("\n[4] Normalizing...")
    normed = normalize_per_sensor(all_records)
    print(f"    {len(normed)} normalized records")

    # Summarize
    print("\n[5] Computing summary...")
    summary = compute_summary(normed)

    # Report
    print("\n[6] Generating report...")
    report = generate_report(summary, config)
    print(report)
    save_report(report, os.path.join(config["OUTPUT_DIR"], "report.txt"))

    # Export
    print("\n[7] Exporting...")
    import csv as csv_mod
    csv_path = os.path.join(config["OUTPUT_DIR"], "cleaned_v2.csv")
    keys = ["timestamp", "sensor_id", "value", "status"]
    with open(csv_path, "w", newline="") as f:
        w = csv_mod.DictWriter(f, fieldnames=keys, extrasaction="ignore")
        w.writeheader()
        w.writerows(normed)
    json_path = os.path.join(config["OUTPUT_DIR"], "summary_v2.json")
    with open(json_path, "w") as f:
        json_mod.dump(summary, f, indent=2)
    print(f"    CSV: {csv_path}")
    print(f"    JSON: {json_path}")
    print("\n" + "=" * 50)
    print("v2.1 PIPELINE COMPLETE")
    print("=" * 50)
    return normed, summary

result_data, result_summary = main_v2(CONFIG)

**Figure 8.2** — Running tests after pipeline

In [ ]:
print("Running all tests...")
test_read()
test_validate()
test_clean()
test_normalize()
test_filters()
test_summary()
print("\nAll tests passed! Pipeline v2.1 is verified.")

---
### ⏱️ Checkpoint 4 of 5 — Regression (target 03:55)

After extending a pipeline, should earlier tests still pass? Answer yes or no.

Enter a short answer in the next cell and run it. Retry after reviewing the
preceding examples if needed.


In [ ]:
checkpoint_4_answer = ""  # enter your answer
check_answer(
    4, checkpoint_4_answer, 'yes',
    'Regression tests protect existing behavior.',
)


---
## Exercises

Complete each exercise in the code cell below it.

### Core Practice and Optional Extension

- **Exercises 1–8:** core in-class practice.
- **Exercises 9 and above:** optional extension; these are not homework.
- Run each completed code cell so Checkpoint 5 can count your local progress.


**EX1:** Add a `SENSOR_FILTER` key to CONFIG that holds a list of sensor IDs to include. Write `filter_sensors(records, ids)` that keeps only those sensors.

In [ ]:
# ✏️ [EX1]
def filter_sensors(records, ids):
    pass


**EX2:** Write `exponential_moving_average(data, alpha)` where each value is `alpha * x + (1-alpha) * prev_ema`.

In [ ]:
# ✏️ [EX2]
def exponential_moving_average(data, alpha=0.3):
    pass


**EX3:** Write `compare_filters_mse(raw, filtered)` that computes Mean Squared Error between raw and filtered.

In [ ]:
# ✏️ [EX3]
def compare_filters_mse(raw, filtered):
    pass


**EX4:** Write `best_window_size(data, max_window=7)` that tests windows 1..max_window and returns the one with smallest MSE against the median filter.

In [ ]:
# ✏️ [EX4]
def best_window_size(data, max_window=7):
    pass


**EX5:** Write 3 new `assert` tests for `moving_average()` edge cases: empty list, single element, window larger than data.

In [ ]:
# ✏️ [EX5]
# Write 3 assert tests for moving_average edge cases


**EX6:** Write `detect_anomalies(records, threshold)` that flags records where value exceeds `mean + threshold * std` for their sensor.

In [ ]:
# ✏️ [EX6]
def detect_anomalies(records, threshold=2.0):
    pass


**EX7:** Write `interpolate_missing(records)` that fills None values with linear interpolation between neighbors.

In [ ]:
# ✏️ [EX7]
def interpolate_missing(records):
    pass


**EX8:** Write `downsample(records, factor)` that keeps every `factor`-th record per sensor.

In [ ]:
# ✏️ [EX8]
def downsample(records, factor=2):
    pass


---
### ⏱️ Checkpoint 5 of 5 — Core Practice (target 04:45)

Run this after Exercises 1–8. It counts only exercise cells that you edited and
ran in this Colab session. It does not inspect correctness or transmit code.


In [ ]:
exercise_checkpoint(5, expected=8)
show_progress_summary()


---
## 🌟 Optional Extension

Exercises 9 and above are optional enrichment. Stop here if the five-hour class has ended.


**EX9:** Write `sensor_correlation(records, sid1, sid2)` that computes Pearson correlation between two sensors' values.

In [ ]:
# ✏️ [EX9]
def sensor_correlation(records, sid1, sid2):
    pass


**EX10:** Write `generate_html_report(summary)` that returns an HTML string with a table of statistics.

In [ ]:
# ✏️ [EX10]
def generate_html_report(summary):
    pass


**EX11:** Write `validate_config(config)` that checks all required keys exist and have valid types.

In [ ]:
# ✏️ [EX11]
def validate_config(config):
    pass


**EX12:** Write `log_pipeline_step(step_name, record_count, log_list)` that appends a timestamped log entry.

In [ ]:
# ✏️ [EX12]
def log_pipeline_step(step_name, record_count, log_list):
    pass


**EX13:** Write `rollback_to_stage(stages_data, stage_name)` that returns the data snapshot from a given stage name.

In [ ]:
# ✏️ [EX13]
def rollback_to_stage(stages_data, stage_name):
    pass


**EX14:** Write `merge_sensor_data(csv1, csv2)` that combines two CSV strings, removes duplicates by timestamp+sensor_id.

In [ ]:
# ✏️ [EX14]
def merge_sensor_data(csv1, csv2):
    pass


**EX15:** Write `pipeline_benchmark(config, n_runs=5)` that runs main_v2 `n_runs` times and prints avg execution time.

In [ ]:
# ✏️ [EX15]
import time
def pipeline_benchmark(config, n_runs=5):
    pass


---
### 🌉 Bridge to Next Week

Your pipeline is now configurable, filtered, and tested. Next week is the final week where you will:
- Prepare a demo of your complete pipeline
- Review code quality and apply refactoring
- Analyze performance and identify bottlenecks
- Reflect on the full semester of learning

Make sure all tests pass before moving on!